In [1]:
"""
=============================================================================
  EMAIL SPAM CLASSIFICATION — PREPROCESSING PIPELINE
  Dataset : Enron Email Dataset (train.jsonl / test.jsonl)
  Tasks   : Cleaning · Tokenization · Stop-word Removal · Lemmatization
            Missing-value Handling · Duplicate Removal · Class Balancing (SMOTE)

=============================================================================
"""

import json
import re
import string
import os
import pickle
import warnings
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

import os, glob

def _find_file(candidates):
    """Auto-detect file by trying multiple common paths."""
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        f"Could not find dataset file. Tried: {candidates}\n"
        "Please set TRAIN_PATH / TEST_PATH manually at the top of this script."
    )

TRAIN_PATH = _find_file([
    "train.jsonl",
    "/content/train.jsonl",
    "1777184744161_train.jsonl",
    "/content/1777184744161_train.jsonl",
])
TEST_PATH = _find_file([
    "test.jsonl",
    "/content/test.jsonl",
    "1777185246170_test.jsonl",
    "/content/1777185246170_test.jsonl",
])
OUTPUT_DIR = "preprocessed_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────
#  1. STOP-WORDS
# ─────────────────────────────────────────────
STOPWORDS = {
    "a","about","above","after","again","against","all","am","an","and","any",
    "are","aren","as","at","be","because","been","before","being","below",
    "between","both","but","by","can","couldn","did","didn","do","does","doesn",
    "doing","don","down","during","each","few","for","from","further","get",
    "got","had","hadn","has","hasn","have","haven","having","he","her","here",
    "hers","herself","him","himself","his","how","i","if","in","into","is",
    "isn","it","its","itself","just","ll","me","mightn","more","most","my",
    "myself","needn","no","nor","not","now","of","off","on","once","only","or",
    "other","our","ours","ourselves","out","over","own","re","s","same","shan",
    "she","should","shouldn","so","some","such","t","than","that","the","their",
    "theirs","them","themselves","then","there","these","they","this","those",
    "through","to","too","under","until","up","us","ve","very","was","wasn",
    "we","were","weren","what","when","where","which","while","who","whom","why",
    "will","with","won","wouldn","you","your","yours","yourself","yourselves",
    "would","could","may","might","shall","has","have","had","said","also",
    "one","two","three","four","five","six","seven","eight","nine","ten",
    "email","mail","message","please","thank","thanks","dear","hi","hello",
    "regards","sincerely","subject","re","fw","fwd","cc","bcc","sent","received",
    "original","wrote","write","forward","forwarded","see","attached","attachment",
}

# ─────────────────────────────────────────────
#  2. SIMPLE LEMMATISER
# ─────────────────────────────────────────────
_SUFFIX_RULES = [
    (r"ational$", "ate"),  (r"tional$",  "tion"), (r"enci$",  "ence"),
    (r"anci$",    "ance"), (r"izer$",    "ize"),  (r"ising$", "ise"),
    (r"izing$",   "ize"),  (r"ised$",    "ise"),  (r"ized$",  "ize"),
    (r"alism$",   "al"),   (r"ness$",    ""),     (r"ment$",  ""),
    (r"fulness$", "ful"),  (r"ousness$", "ous"),  (r"iveness$","ive"),
    (r"ation$",   "ate"),  (r"ator$",    "ate"),  (r"alism$", "al"),
    (r"ies$",     "y"),    (r"ied$",     "y"),    (r"ing$",   ""),
    (r"edly$",    ""),     (r"ingly$",   ""),     (r"ful$",   ""),
    (r"ous$",     ""),     (r"ive$",     ""),     (r"ble$",   ""),
    (r"tion$",    "te"),   (r"sses$",    "ss"),   (r"s$",     ""),
    (r"ed$",      ""),
]

def lemmatize(word: str) -> str:
    """Lightweight rule-based lemmatizer (Porter-inspired)."""
    if len(word) <= 3:
        return word
    for pattern, replacement in _SUFFIX_RULES:
        if re.search(pattern, word):
            stem = re.sub(pattern, replacement, word)
            if len(stem) >= 3:
                return stem
    return word

# ─────────────────────────────────────────────
#  3. LOAD DATA
# ─────────────────────────────────────────────
def load_jsonl(path: str) -> pd.DataFrame:
    """Load a JSONL file, skipping any malformed / truncated lines."""
    records = []
    skipped = 0
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                skipped += 1
    if skipped:
        print(f"  WARNING: Skipped {skipped} malformed line(s) in: {path}")
    return pd.DataFrame(records)

print("=" * 65)
print("  STEP 1 — LOADING DATA")
print("=" * 65)

df_train = load_jsonl(TRAIN_PATH)
df_test  = load_jsonl(TEST_PATH)

print(f"  Train shape : {df_train.shape}")
print(f"  Test  shape : {df_test.shape}")
print(f"  Columns     : {list(df_train.columns)}")

# ─────────────────────────────────────────────
#  4. INITIAL DATA INSPECTION
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 2 — DATA INSPECTION")
print("=" * 65)

for name, df in [("TRAIN", df_train), ("TEST", df_test)]:
    label_counts = df["label"].value_counts()
    total = len(df)
    spam  = label_counts.get(1, 0)
    ham   = label_counts.get(0, 0)
    ratio = spam / ham if ham > 0 else float("inf")
    print(f"\n  [{name}]")
    print(f"    Total rows   : {total}")
    print(f"    Ham  (0)     : {ham}  ({ham/total*100:.1f}%)")
    print(f"    Spam (1)     : {spam}  ({spam/total*100:.1f}%)")
    print(f"    Spam:Ham     : {ratio:.2f}")
    print(f"    Missing text : {df['text'].isnull().sum()}")
    print(f"    Duplicates   : {df['text'].duplicated().sum()}")
    empty = (df["text"].str.strip() == "") | (df["text"].str.len() == 0)
    print(f"    Empty text   : {empty.sum()}")

# ─────────────────────────────────────────────
#  5. MISSING VALUE HANDLING
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 3 — MISSING VALUE HANDLING")
print("=" * 65)

def handle_missing(df: pd.DataFrame, name: str) -> pd.DataFrame:
    before = len(df)

    # Drop rows where text is NaN
    df = df.dropna(subset=["text"])

    # Fill NaN subject with empty string then merge into text if subject missing
    df["subject"] = df["subject"].fillna("")
    df["message"] = df["message"].fillna("")

    # Rebuild 'text' from subject + message if text is empty/whitespace after NaN drop
    mask_empty = df["text"].str.strip() == ""
    df.loc[mask_empty, "text"] = (
        df.loc[mask_empty, "subject"] + " " + df.loc[mask_empty, "message"]
    ).str.strip()

    # After rebuild, drop rows still empty
    still_empty = df["text"].str.strip() == ""
    dropped_empty = still_empty.sum()
    df = df[~still_empty]

    after = len(df)
    print(f"  [{name}] rows before: {before} → after: {after}  (removed {before-after})")
    return df.reset_index(drop=True)

df_train = handle_missing(df_train, "TRAIN")
df_test  = handle_missing(df_test,  "TEST")

# ─────────────────────────────────────────────
#  6. DUPLICATE REMOVAL
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 4 — DUPLICATE REMOVAL")
print("=" * 65)

def remove_duplicates(df: pd.DataFrame, name: str) -> pd.DataFrame:
    before = len(df)
    df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
    after = len(df)
    print(f"  [{name}] duplicates removed: {before - after}  | rows left: {after}")
    return df

df_train = remove_duplicates(df_train, "TRAIN")
df_test  = remove_duplicates(df_test,  "TEST")

# ─────────────────────────────────────────────
#  7. TEXT CLEANING FUNCTION
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 5 — TEXT CLEANING  (NLP Preprocessing)")
print("=" * 65)

def clean_text(text: str) -> str:
    """
    Full NLP preprocessing pipeline:
      1. Lowercase
      2. Remove HTML / XML tags
      3. Remove URLs
      4. Remove email addresses
      5. Remove phone numbers
      6. Remove special chars & punctuation
      7. Remove digits
      8. Collapse whitespace
      9. Tokenize (regex word boundaries)
     10. Remove stop-words
     11. Remove very short tokens (len < 2)
     12. Lemmatize
    """
    if not isinstance(text, str):
        return ""

    # 1. Lowercase
    text = text.lower()

    # 2. Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # 3. Remove URLs
    text = re.sub(r"http\S+|www\.\S+|https\S+", " ", text)

    # 4. Remove email addresses
    text = re.sub(r"\S+@\S+", " ", text)

    # 5. Remove phone numbers
    text = re.sub(r"\b\d[\d\s\-\.\(\)]{6,}\d\b", " ", text)

    # 6. Remove special characters & punctuation (keep letters & spaces)
    text = re.sub(r"[^a-z\s]", " ", text)

    # 7. Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # 8. Tokenize
    tokens = re.findall(r"\b[a-z]{2,}\b", text)

    # 9. Remove stop-words + very short tokens
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 2]

    # 10. Lemmatize
    tokens = [lemmatize(t) for t in tokens]

    return " ".join(tokens)


# Apply cleaning
print("  Applying cleaning pipeline to TRAIN set ...")
df_train["clean_text"] = df_train["text"].apply(clean_text)
print("  Applying cleaning pipeline to TEST set ...")
df_test["clean_text"]  = df_test["text"].apply(clean_text)

# Drop rows where clean_text is empty after cleaning
before_t = len(df_train)
df_train = df_train[df_train["clean_text"].str.strip() != ""].reset_index(drop=True)
before_e = len(df_test)
df_test  = df_test[df_test["clean_text"].str.strip()  != ""].reset_index(drop=True)

print(f"  TRAIN rows after cleaning filter: {before_t} → {len(df_train)}")
print(f"  TEST  rows after cleaning filter: {before_e} → {len(df_test)}")

# Sample preview
print("\n  ── Sample cleaned texts ──")
for i in range(2):
    row = df_train.iloc[i]
    print(f"\n  [{row['label_text'].upper()}] ORIGINAL (first 120):\n  {row['text'][:120]}")
    print(f"  CLEANED  (first 120):\n  {row['clean_text'][:120]}")

# Token stats
df_train["token_count"] = df_train["clean_text"].str.split().str.len()
df_test["token_count"]  = df_test["clean_text"].str.split().str.len()
print(f"\n  TRAIN token count — mean: {df_train['token_count'].mean():.1f}  "
      f"| median: {df_train['token_count'].median():.0f}  "
      f"| max: {df_train['token_count'].max()}")

# ─────────────────────────────────────────────
#  8. LABEL ENCODING
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 6 — LABEL ENCODING")
print("=" * 65)

# Labels are already numeric (0=ham, 1=spam) — verify and use directly
assert set(df_train["label"].unique()).issubset({0, 1}), "Unexpected label values!"
assert set(df_test["label"].unique()).issubset({0, 1}),  "Unexpected label values!"

y_train = df_train["label"].values
y_test  = df_test["label"].values

print(f"  Train — ham: {(y_train==0).sum()}  spam: {(y_train==1).sum()}")
print(f"  Test  — ham: {(y_test==0).sum()}   spam: {(y_test==1).sum()}")

# ─────────────────────────────────────────────
#  9. TEXT TO NUMERICAL FEATURES (Simple CountVectorizer for SMOTE)
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 7 — TEXT TO NUMERICAL FEATURES (for SMOTE)")
print("=" * 65)

# Using CountVectorizer as simple numerical representation for SMOTE
from sklearn.feature_extraction.text import CountVectorizer

count_vec = CountVectorizer(
    max_features=5000,  # Limited features for SMOTE
    ngram_range=(1, 1),
    min_df=2,
    max_df=0.95,
)

print("  Fitting CountVectorizer on TRAIN clean text ...")
X_train_count = count_vec.fit_transform(df_train["clean_text"])
X_test_count = count_vec.transform(df_test["clean_text"])

print(f"  X_train_count shape : {X_train_count.shape}")
print(f"  X_test_count  shape : {X_test_count.shape}")
print(f"  Vocabulary size      : {len(count_vec.vocabulary_)}")

# ─────────────────────────────────────────────
#  10. CLASS BALANCE CHECK + SMOTE
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 8 — CLASS BALANCING (SMOTE)")
print("=" * 65)

spam_count = (y_train == 1).sum()
ham_count  = (y_train == 0).sum()
imbalance_ratio = max(spam_count, ham_count) / min(spam_count, ham_count)

print(f"  Before SMOTE — ham: {ham_count}  spam: {spam_count}  ratio: {imbalance_ratio:.3f}")

if imbalance_ratio > 1.5:
    print("  Imbalance detected (ratio > 1.5) — applying SMOTE ...")
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train_count, y_train)
    new_spam = (y_train_balanced == 1).sum()
    new_ham  = (y_train_balanced == 0).sum()
    print(f"  After  SMOTE — ham: {new_ham}  spam: {new_spam}  ratio: {new_ham/new_spam:.3f}")
else:
    print(f"  Dataset is sufficiently balanced (ratio {imbalance_ratio:.3f} ≤ 1.5) — skipping SMOTE")
    X_train_balanced = X_train_count
    y_train_balanced = y_train

# ─────────────────────────────────────────────
#  11. SAVE PREPROCESSED ARTIFACTS
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 9 — SAVING OUTPUTS")
print("=" * 65)

# Save cleaned DataFrames as CSV
df_train.to_csv(f"{OUTPUT_DIR}/train_cleaned.csv", index=False)
df_test.to_csv(f"{OUTPUT_DIR}/test_cleaned.csv",   index=False)
print(f"  Saved cleaned CSVs → {OUTPUT_DIR}/train_cleaned.csv & test_cleaned.csv")

# Create balanced dataset from SMOTE results
# Convert sparse matrix to dense and create DataFrame
X_train_balanced_dense = X_train_balanced.toarray()
feature_names = count_vec.get_feature_names_out()

# Create balanced train DataFrame with features
df_train_balanced = pd.DataFrame(X_train_balanced_dense, columns=feature_names)
df_train_balanced['label'] = y_train_balanced

# Add clean_text back to balanced dataset by matching with original
# Since SMOTE generates synthetic samples, we'll include all original samples + synthetic
# For simplicity, save the feature representation with labels
df_train_balanced.to_csv(f"{OUTPUT_DIR}/train_balanced.csv", index=False)
print(f"  Saved SMOTE-balanced dataset → {OUTPUT_DIR}/train_balanced.csv")

# Save CountVectorizer
with open(f"{OUTPUT_DIR}/count_vectorizer.pkl", "wb") as f:
    pickle.dump(count_vec, f)
print(f"  Saved CountVectorizer → {OUTPUT_DIR}/count_vectorizer.pkl")

# Save numerical matrices (sparse)
import scipy.sparse as sp
sp.save_npz(f"{OUTPUT_DIR}/X_train_count.npz",      X_train_count)
sp.save_npz(f"{OUTPUT_DIR}/X_train_balanced.npz",   X_train_balanced)
sp.save_npz(f"{OUTPUT_DIR}/X_test_count.npz",       X_test_count)
np.save(f"{OUTPUT_DIR}/y_train.npy",          y_train)
np.save(f"{OUTPUT_DIR}/y_train_balanced.npy", y_train_balanced)
np.save(f"{OUTPUT_DIR}/y_test.npy",           y_test)
print(f"  Saved numerical matrices & labels → {OUTPUT_DIR}/")

# Save vocabulary
vocab_df = pd.DataFrame(
    sorted(count_vec.vocabulary_.items(), key=lambda x: x[1]),
    columns=["token", "index"]
)
vocab_df.to_csv(f"{OUTPUT_DIR}/vocabulary.csv", index=False)
print(f"  Saved vocabulary ({len(vocab_df)} tokens) → {OUTPUT_DIR}/vocabulary.csv")

# ─────────────────────────────────────────────
#  12. FINAL SUMMARY
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  PREPROCESSING COMPLETE — SUMMARY")
print("=" * 65)
print(f"""
  ┌──────────────────────────────────────────────────────────┐
  │  Dataset          : Enron Email (JSONL)                  │
  │  Train rows       : {len(df_train):<8}  (after cleaning)         │
  │  Test  rows       : {len(df_test):<8}  (after cleaning)         │
  │  Missing values   : Handled (filled / dropped)           │
  │  Duplicates       : Removed                              │
  │  Text cleaning    : lowercase, HTML/URL/email strip,     │
  │                     punct removal, stop-word removal,    │
  │                     lemmatization                        │
  │  Features         : CountVectorizer (for SMOTE)         │
  │  Vocab size       : {len(count_vec.vocabulary_):<8}                        │
  │  X_train shape    : {str(X_train_count.shape):<20}               │
  │  X_test  shape    : {str(X_test_count.shape):<20}               │
  │  Balanced X_train : {str(X_train_balanced.shape):<20}               │
  │  Class balance    : {'SMOTE applied' if imbalance_ratio > 1.5 else 'Already balanced':<30}       │
  │  Output dir       : {OUTPUT_DIR:<20}                    │
  └──────────────────────────────────────────────────────────┘

  Output files saved:
    → train_cleaned.csv   : Cleaned training data
    → test_cleaned.csv    : Cleaned test data
    → train_balanced.csv  : SMOTE-balanced training data
    → vocabulary.csv      : Feature vocabulary
    → count_vectorizer.pkl: Vectorizer for transform
""")


  STEP 1 — LOADING DATA
  Train shape : (6673, 7)
  Test  shape : (2000, 7)
  Columns     : ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date']

  STEP 2 — DATA INSPECTION

  [TRAIN]
    Total rows   : 6673
    Ham  (0)     : 3318  (49.7%)
    Spam (1)     : 3355  (50.3%)
    Spam:Ham     : 1.01
    Missing text : 0
    Duplicates   : 124
    Empty text   : 8

  [TEST]
    Total rows   : 2000
    Ham  (0)     : 992  (49.6%)
    Spam (1)     : 1008  (50.4%)
    Spam:Ham     : 1.02
    Missing text : 0
    Duplicates   : 18
    Empty text   : 0

  STEP 3 — MISSING VALUE HANDLING
  [TRAIN] rows before: 6673 → after: 6665  (removed 8)
  [TEST] rows before: 2000 → after: 2000  (removed 0)

  STEP 4 — DUPLICATE REMOVAL
  [TRAIN] duplicates removed: 117  | rows left: 6548
  [TEST] duplicates removed: 18  | rows left: 1982

  STEP 5 — TEXT CLEANING  (NLP Preprocessing)
  Applying cleaning pipeline to TRAIN set ...
  Applying cleaning pipeline to TEST set ...
  TRAIN row

In [4]:

%pip install gensim

"""
=============================================================================
  EMAIL SPAM CLASSIFICATION — FEATURE EXTRACTION PIPELINE
  Project #3 : Email Spam Classification Using NLP and Machine Learning

  Input  : Preprocessed artifacts from Preprocess.ipynb
           - train_cleaned.csv / test_cleaned.csv   (cleaned text + labels)
           - X_train_count.npz / X_test_count.npz   (CountVectorizer matrix)
           - count_vectorizer.pkl                    (fitted CountVectorizer)
           - y_train.npy / y_test.npy                (labels)
           - y_train_balanced.npy                    (SMOTE-balanced labels)
           - vocabulary.csv                          (token → index mapping)

  Extractions:
    1. TF-IDF  (unigram + bigram, up to 10 000 features)
    2. TF-IDF  (character n-grams 2-4, up to 5 000 features)
    3. Word Embeddings  — mean-pooled GloVe-style vectors
       (falls back to a trained Word2Vec if GloVe file absent)
    4. Combined feature matrix  (TF-IDF + embeddings)

  Outputs  → feature_extraction_output/
    - X_train_tfidf.npz            TF-IDF word n-gram (sparse)
    - X_test_tfidf.npz
    - X_train_tfidf_char.npz       TF-IDF char n-gram (sparse)
    - X_test_tfidf_char.npz
    - X_train_embeddings.npy       Mean-pooled word embeddings (dense)
    - X_test_embeddings.npy
    - X_train_combined.npz         TF-IDF (word) + embeddings stacked
    - X_test_combined.npz
    - tfidf_vectorizer.pkl         Fitted word TF-IDF vectorizer
    - tfidf_char_vectorizer.pkl    Fitted char TF-IDF vectorizer
    - feature_summary.txt          Human-readable summary

=============================================================================
"""

import os
import re
import json
import pickle
import warnings
import logging
import numpy as np
import pandas as pd
import scipy.sparse as sp

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")

# ── Optional: gensim for Word2Vec fallback ────────────────────────────────
try:
    from gensim.models import Word2Vec
    GENSIM_AVAILABLE = True
except ImportError:
    GENSIM_AVAILABLE = False

# ─────────────────────────────────────────────────────────────────────────
#  CONFIGURATION  — edit paths here if your files live elsewhere
# ─────────────────────────────────────────────────────────────────────────

INPUT_DIR  = "preprocessed_output"   # folder produced by Preprocess.ipynb
OUTPUT_DIR = "feature_extraction_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Optional: path to a pre-trained GloVe text file (e.g. glove.6B.100d.txt)
# If not found, the script trains a lightweight Word2Vec on the corpus instead.
GLOVE_PATH   = os.environ.get("GLOVE_PATH", "glove.6B.100d.txt")
EMBED_DIM    = 100          # must match GloVe file if using GloVe

# TF-IDF hyper-parameters
TFIDF_WORD_MAX_FEATURES = 10_000
TFIDF_CHAR_MAX_FEATURES = 5_000
TFIDF_WORD_NGRAM        = (1, 2)   # unigrams + bigrams
TFIDF_CHAR_NGRAM        = (2, 4)   # character n-grams 2-4
TFIDF_MIN_DF            = 2
TFIDF_MAX_DF            = 0.95
TFIDF_SUBLINEAR_TF      = True     # log(tf) + 1 smoothing

# Word2Vec hyper-params (fallback when GloVe absent)
W2V_VECTOR_SIZE = EMBED_DIM
W2V_WINDOW      = 5
W2V_MIN_COUNT   = 2
W2V_WORKERS     = 4
W2V_EPOCHS      = 5

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="  %(message)s",
)
log = logging.getLogger(__name__)

# ─────────────────────────────────────────────────────────────────────────
#  HELPERS
# ─────────────────────────────────────────────────────────────────────────

def _find(candidates):
    """Return the first existing path from a list of candidates."""
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        f"Could not find any of: {candidates}\n"
        f"Make sure INPUT_DIR='{INPUT_DIR}' points to your preprocessed_output folder."
    )


def load_preprocessed():
    """Load all artefacts produced by the preprocessing notebook."""
    log.info("Loading preprocessed artefacts ...")

    train_csv = _find([
        f"{INPUT_DIR}/train_cleaned.csv",
        "train_cleaned.csv",
    ])
    test_csv = _find([
        f"{INPUT_DIR}/test_cleaned.csv",
        "test_cleaned.csv",
    ])

    df_train = pd.read_csv(train_csv)
    df_test  = pd.read_csv(test_csv)

    y_train = np.load(_find([f"{INPUT_DIR}/y_train.npy", "y_train.npy"]))
    y_test  = np.load(_find([f"{INPUT_DIR}/y_test.npy",  "y_test.npy"]))

    y_train_balanced = np.load(_find([
        f"{INPUT_DIR}/y_train_balanced.npy", "y_train_balanced.npy"
    ]))

    X_train_count = sp.load_npz(_find([
        f"{INPUT_DIR}/X_train_count.npz", "X_train_count.npz"
    ]))
    X_test_count = sp.load_npz(_find([
        f"{INPUT_DIR}/X_test_count.npz", "X_test_count.npz"
    ]))

    with open(_find([
        f"{INPUT_DIR}/count_vectorizer.pkl", "count_vectorizer.pkl"
    ]), "rb") as fh:
        count_vec = pickle.load(fh)

    log.info(f"Train rows : {len(df_train)}  |  Test rows : {len(df_test)}")
    log.info(f"X_train_count shape : {X_train_count.shape}")
    log.info(f"X_test_count  shape : {X_test_count.shape}")
    log.info(f"y_train dist  — ham: {(y_train==0).sum()}  spam: {(y_train==1).sum()}")
    log.info(f"y_test  dist  — ham: {(y_test==0).sum()}   spam: {(y_test==1).sum()}")

    return (
        df_train, df_test,
        X_train_count, X_test_count,
        y_train, y_test, y_train_balanced,
        count_vec,
    )


# ─────────────────────────────────────────────────────────────────────────
#  FEATURE BLOCK 1 — TF-IDF  (word n-grams)
# ─────────────────────────────────────────────────────────────────────────

def extract_tfidf_word(train_texts, test_texts):
    """
    Fit a TF-IDF vectorizer on training data and transform both splits.

    Features:
      - analyzer  : word
      - n-gram     : (1, 2)  — unigrams + bigrams
      - max_features : 10 000
      - sublinear_tf : True  (log(tf) smoothing)
      - norm         : 'l2'

    Returns sparse matrices X_train, X_test and the fitted vectorizer.
    """
    print("\n" + "=" * 65)
    print("  FEATURE BLOCK 1 — TF-IDF  (word unigrams + bigrams)")
    print("=" * 65)

    tfidf = TfidfVectorizer(
        analyzer      = "word",
        ngram_range   = TFIDF_WORD_NGRAM,
        max_features  = TFIDF_WORD_MAX_FEATURES,
        min_df        = TFIDF_MIN_DF,
        max_df        = TFIDF_MAX_DF,
        sublinear_tf  = TFIDF_SUBLINEAR_TF,
        norm          = "l2",
        token_pattern = r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
    )

    log.info("Fitting TF-IDF word vectorizer on TRAIN ...")
    X_train = tfidf.fit_transform(train_texts)
    X_test  = tfidf.transform(test_texts)

    vocab_size = len(tfidf.vocabulary_)
    log.info(f"Vocabulary size   : {vocab_size}")
    log.info(f"X_train_tfidf     : {X_train.shape}  (sparse, nnz={X_train.nnz})")
    log.info(f"X_test_tfidf      : {X_test.shape}")

    # Top-10 features by mean TF-IDF score on train
    mean_tfidf = np.asarray(X_train.mean(axis=0)).ravel()
    top_idx    = mean_tfidf.argsort()[::-1][:10]
    inv_vocab  = {v: k for k, v in tfidf.vocabulary_.items()}
    top_tokens = [inv_vocab[i] for i in top_idx]
    log.info(f"Top-10 TF-IDF tokens: {top_tokens}")

    return X_train, X_test, tfidf


# ─────────────────────────────────────────────────────────────────────────
#  FEATURE BLOCK 2 — TF-IDF  (character n-grams)
# ─────────────────────────────────────────────────────────────────────────

def extract_tfidf_char(train_texts, test_texts):
    """
    Character-level TF-IDF captures morphological patterns and is robust
    against deliberate misspellings used in spam (e.g. "v1agra", "fr33").

    Features:
      - analyzer    : char_wb  (char n-grams within word boundaries)
      - n-gram       : (2, 4)
      - max_features : 5 000
    """
    print("\n" + "=" * 65)
    print("  FEATURE BLOCK 2 — TF-IDF  (character n-grams 2-4)")
    print("=" * 65)

    tfidf_char = TfidfVectorizer(
        analyzer     = "char_wb",
        ngram_range  = TFIDF_CHAR_NGRAM,
        max_features = TFIDF_CHAR_MAX_FEATURES,
        min_df       = TFIDF_MIN_DF,
        max_df       = TFIDF_MAX_DF,
        sublinear_tf = TFIDF_SUBLINEAR_TF,
        norm         = "l2",
    )

    log.info("Fitting TF-IDF char vectorizer on TRAIN ...")
    X_train = tfidf_char.fit_transform(train_texts)
    X_test  = tfidf_char.transform(test_texts)

    log.info(f"Char vocab size   : {len(tfidf_char.vocabulary_)}")
    log.info(f"X_train_tfidf_char: {X_train.shape}  (sparse, nnz={X_train.nnz})")
    log.info(f"X_test_tfidf_char : {X_test.shape}")

    return X_train, X_test, tfidf_char


# ─────────────────────────────────────────────────────────────────────────
#  FEATURE BLOCK 3 — Word Embeddings
# ─────────────────────────────────────────────────────────────────────────

def _load_glove(path, embed_dim):
    """Parse a GloVe text file into a {word: np.ndarray} dictionary."""
    log.info(f"Loading GloVe vectors from '{path}' ...")
    embeddings = {}
    with open(path, "r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            parts = line.rstrip().split(" ")
            if len(parts) != embed_dim + 1:
                continue
            word = parts[0]
            vec  = np.array(parts[1:], dtype=np.float32)
            embeddings[word] = vec
    log.info(f"Loaded {len(embeddings):,} GloVe vectors  (dim={embed_dim})")
    return embeddings


def _train_word2vec(corpus_tokens, embed_dim):
    """Train a Word2Vec model on the tokenized corpus."""
    if not GENSIM_AVAILABLE:
        raise ImportError(
            "gensim is required for Word2Vec training.\n"
            "Install it with:  pip install gensim\n"
            "Or supply a GloVe file at GLOVE_PATH."
        )
    log.info(f"Training Word2Vec  (dim={embed_dim}, window={W2V_WINDOW}, "
             f"epochs={W2V_EPOCHS}) ...")
    model = Word2Vec(
        sentences   = corpus_tokens,
        vector_size = embed_dim,
        window      = W2V_WINDOW,
        min_count   = W2V_MIN_COUNT,
        workers      = W2V_WORKERS,
        epochs      = W2V_EPOCHS,
        seed        = 42,
    )
    log.info(f"Word2Vec vocabulary : {len(model.wv):,} words")
    # Return a simple dict-like interface identical to GloVe lookup
    return {w: model.wv[w] for w in model.wv.index_to_key}


def _mean_pool(texts, embeddings, embed_dim):
    """
    For each document, compute the mean of all token embedding vectors.
    Tokens absent from the embedding space are silently skipped.
    Documents with no known tokens receive a zero vector.
    """
    matrix = np.zeros((len(texts), embed_dim), dtype=np.float32)
    oov_total = 0
    token_total = 0

    for i, text in enumerate(texts):
        tokens = text.split() if isinstance(text, str) else []
        vecs   = []
        for tok in tokens:
            token_total += 1
            if tok in embeddings:
                vecs.append(embeddings[tok])
            else:
                oov_total += 1
        if vecs:
            matrix[i] = np.mean(vecs, axis=0)

    oov_rate = oov_total / token_total * 100 if token_total > 0 else 0
    log.info(f"OOV rate : {oov_rate:.1f}%  ({oov_total:,} / {token_total:,} tokens)")
    return matrix


def extract_embeddings(train_texts, test_texts, embed_dim=EMBED_DIM):
    """
    Build dense mean-pooled embedding matrices for train and test.

    Strategy (in priority order):
      1. Load GloVe from GLOVE_PATH if the file exists.
      2. Otherwise train a lightweight Word2Vec on the training corpus.

    The final vectors are L2-normalised row-wise.

    Returns dense numpy arrays X_train_emb (n_train × embed_dim),
                                X_test_emb  (n_test  × embed_dim).
    """
    print("\n" + "=" * 65)
    print("  FEATURE BLOCK 3 — Word Embeddings  (mean-pooled)")
    print("=" * 65)

    # --- Obtain embedding lookup -----------------------------------------
    if os.path.exists(GLOVE_PATH):
        log.info("GloVe file found — using pre-trained GloVe vectors.")
        embeddings = _load_glove(GLOVE_PATH, embed_dim)
    else:
        log.info(
            f"GloVe file not found at '{GLOVE_PATH}'.\n"
            "  Falling back to training Word2Vec on the training corpus."
        )
        corpus_tokens = [
            t.split() for t in train_texts if isinstance(t, str)
        ]
        embeddings = _train_word2vec(corpus_tokens, embed_dim)

    # --- Mean-pool ----------------------------------------------------------
    log.info("Building train embedding matrix ...")
    X_train_emb = _mean_pool(train_texts.tolist(), embeddings, embed_dim)

    log.info("Building test embedding matrix ...")
    X_test_emb  = _mean_pool(test_texts.tolist(),  embeddings, embed_dim)

    # --- L2 normalise -------------------------------------------------------
    X_train_emb = normalize(X_train_emb, norm="l2")
    X_test_emb  = normalize(X_test_emb,  norm="l2")

    log.info(f"X_train_embeddings : {X_train_emb.shape}  (dense)")
    log.info(f"X_test_embeddings  : {X_test_emb.shape}")

    return X_train_emb, X_test_emb


# ─────────────────────────────────────────────────────────────────────────
#  FEATURE BLOCK 4 — Combined  (TF-IDF word + Embeddings)
# ─────────────────────────────────────────────────────────────────────────

def combine_features(X_tfidf_train, X_tfidf_test,
                     X_emb_train,   X_emb_test):
    """
    Horizontally stack sparse TF-IDF features with dense embedding
    features to produce a single combined feature matrix.

    TF-IDF  (sparse, n × 10 000)
    Embeddings (dense, n × 100)
    → Combined (sparse, n × 10 100)
    """
    print("\n" + "=" * 65)
    print("  FEATURE BLOCK 4 — Combined  (TF-IDF + Embeddings)")
    print("=" * 65)

    # Convert dense embeddings to sparse for horizontal stack
    X_emb_train_sp = sp.csr_matrix(X_emb_train)
    X_emb_test_sp  = sp.csr_matrix(X_emb_test)

    X_train_combined = sp.hstack([X_tfidf_train, X_emb_train_sp], format="csr")
    X_test_combined  = sp.hstack([X_tfidf_test,  X_emb_test_sp],  format="csr")

    log.info(f"X_train_combined : {X_train_combined.shape}")
    log.info(f"X_test_combined  : {X_test_combined.shape}")

    return X_train_combined, X_test_combined


# ─────────────────────────────────────────────────────────────────────────
#  SAVE OUTPUTS
# ─────────────────────────────────────────────────────────────────────────

def save_outputs(
    X_train_tfidf,      X_test_tfidf,       tfidf_vec,
    X_train_tfidf_char, X_test_tfidf_char,  tfidf_char_vec,
    X_train_emb,        X_test_emb,
    X_train_combined,   X_test_combined,
    y_train,            y_test,             y_train_balanced,
):
    print("\n" + "=" * 65)
    print("  SAVING FEATURE EXTRACTION OUTPUTS")
    print("=" * 65)

    out = OUTPUT_DIR

    # Sparse TF-IDF matrices
    sp.save_npz(f"{out}/X_train_tfidf.npz",      X_train_tfidf)
    sp.save_npz(f"{out}/X_test_tfidf.npz",       X_test_tfidf)
    sp.save_npz(f"{out}/X_train_tfidf_char.npz", X_train_tfidf_char)
    sp.save_npz(f"{out}/X_test_tfidf_char.npz",  X_test_tfidf_char)
    sp.save_npz(f"{out}/X_train_combined.npz",   X_train_combined)
    sp.save_npz(f"{out}/X_test_combined.npz",    X_test_combined)
    log.info("Saved sparse matrices  (.npz)")

    # Dense embedding matrices
    np.save(f"{out}/X_train_embeddings.npy", X_train_emb)
    np.save(f"{out}/X_test_embeddings.npy",  X_test_emb)
    log.info("Saved dense embedding matrices  (.npy)")

    # Labels (also copy for convenience)
    np.save(f"{out}/y_train.npy",          y_train)
    np.save(f"{out}/y_test.npy",           y_test)
    np.save(f"{out}/y_train_balanced.npy", y_train_balanced)
    log.info("Saved label arrays  (.npy)")

    # Fitted vectorizers
    with open(f"{out}/tfidf_vectorizer.pkl",      "wb") as fh:
        pickle.dump(tfidf_vec,      fh)
    with open(f"{out}/tfidf_char_vectorizer.pkl", "wb") as fh:
        pickle.dump(tfidf_char_vec, fh)
    log.info("Saved fitted TF-IDF vectorizers  (.pkl)")

    # Human-readable summary
    summary_lines = [
        "=" * 65,
        "  FEATURE EXTRACTION SUMMARY",
        "=" * 65,
        f"  Output directory    : {os.path.abspath(out)}",
        "",
        "  ── TF-IDF Word Features ──",
        f"  Vectorizer          : word, ngram=(1,2), max_features={TFIDF_WORD_MAX_FEATURES}",
        f"  X_train_tfidf       : {X_train_tfidf.shape}",
        f"  X_test_tfidf        : {X_test_tfidf.shape}",
        "",
        "  ── TF-IDF Char Features ──",
        f"  Vectorizer          : char_wb, ngram=(2,4), max_features={TFIDF_CHAR_MAX_FEATURES}",
        f"  X_train_tfidf_char  : {X_train_tfidf_char.shape}",
        f"  X_test_tfidf_char   : {X_test_tfidf_char.shape}",
        "",
        "  ── Word Embeddings (mean-pooled) ──",
        f"  Embedding dim       : {X_train_emb.shape[1]}",
        f"  Source              : {'GloVe' if os.path.exists(GLOVE_PATH) else 'Word2Vec (trained on corpus)'}",
        f"  Normalisation       : L2",
        f"  X_train_embeddings  : {X_train_emb.shape}",
        f"  X_test_embeddings   : {X_test_emb.shape}",
        "",
        "  ── Combined Features (TF-IDF + Embeddings) ──",
        f"  X_train_combined    : {X_train_combined.shape}",
        f"  X_test_combined     : {X_test_combined.shape}",
        "",
        "  ── Labels ──",
        f"  y_train             : {y_train.shape}  — ham: {(y_train==0).sum()}  spam: {(y_train==1).sum()}",
        f"  y_test              : {y_test.shape}   — ham: {(y_test==0).sum()}   spam: {(y_test==1).sum()}",
        f"  y_train_balanced    : {y_train_balanced.shape}",
        "",
        "  ── Saved Files ──",
        f"  {out}/X_train_tfidf.npz",
        f"  {out}/X_test_tfidf.npz",
        f"  {out}/X_train_tfidf_char.npz",
        f"  {out}/X_test_tfidf_char.npz",
        f"  {out}/X_train_embeddings.npy",
        f"  {out}/X_test_embeddings.npy",
        f"  {out}/X_train_combined.npz",
        f"  {out}/X_test_combined.npz",
        f"  {out}/tfidf_vectorizer.pkl",
        f"  {out}/tfidf_char_vectorizer.pkl",
        f"  {out}/y_train.npy",
        f"  {out}/y_test.npy",
        f"  {out}/y_train_balanced.npy",
        "=" * 65,
    ]

    summary_text = "\n".join(summary_lines)
    with open(f"{out}/feature_summary.txt", "w") as fh:
        fh.write(summary_text)

    print("\n" + summary_text)


# ─────────────────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────────────────

def main():
    print("=" * 65)
    print("  EMAIL SPAM CLASSIFICATION — FEATURE EXTRACTION")
    print("=" * 65)

    # ── 0. Load preprocessed data ─────────────────────────────────────────
    (
        df_train, df_test,
        X_train_count, X_test_count,
        y_train, y_test, y_train_balanced,
        count_vec,
    ) = load_preprocessed()

    train_texts = df_train["clean_text"].fillna("")
    test_texts  = df_test["clean_text"].fillna("")

    # ── 1. TF-IDF word n-grams ────────────────────────────────────────────
    X_train_tfidf, X_test_tfidf, tfidf_vec = extract_tfidf_word(
        train_texts, test_texts
    )

    # ── 2. TF-IDF character n-grams ───────────────────────────────────────
    X_train_tfidf_char, X_test_tfidf_char, tfidf_char_vec = extract_tfidf_char(
        train_texts, test_texts
    )

    # ── 3. Word embeddings (GloVe or Word2Vec) ────────────────────────────
    X_train_emb, X_test_emb = extract_embeddings(train_texts, test_texts)

    # ── 4. Combined feature matrix ────────────────────────────────────────
    X_train_combined, X_test_combined = combine_features(
        X_train_tfidf, X_test_tfidf,
        X_train_emb,   X_test_emb,
    )

    # ── 5. Save everything ────────────────────────────────────────────────
    save_outputs(
        X_train_tfidf,      X_test_tfidf,       tfidf_vec,
        X_train_tfidf_char, X_test_tfidf_char,  tfidf_char_vec,
        X_train_emb,        X_test_emb,
        X_train_combined,   X_test_combined,
        y_train,            y_test,             y_train_balanced,
    )

    print("\n  ✓ Feature extraction complete.")
    print(f"  All outputs saved to: {os.path.abspath(OUTPUT_DIR)}\n")


# ─────────────────────────────────────────────────────────────────────────
#  USAGE NOTES
# ─────────────────────────────────────────────────────────────────────────
# Run:
#   python feature_extraction.py
#
# Prerequisites:
#   pip install scikit-learn numpy pandas scipy
#   pip install gensim          # only if GloVe file is absent
#
# Optional GloVe download (100-dim, 822 MB):
#   wget http://nlp.stanford.edu/data/glove.6B.zip
#   unzip glove.6B.zip
#   export GLOVE_PATH=glove.6B.100d.txt
#
# INPUT_DIR must contain the 9 artefacts produced by Preprocess.ipynb:
#   train_cleaned.csv, test_cleaned.csv,
#   X_train_count.npz, X_test_count.npz,
#   y_train.npy, y_test.npy, y_train_balanced.npy,
#   count_vectorizer.pkl, vocabulary.csv
# ─────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    main()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 45.2 MB/s eta 0:00:00
  EMAIL SPAM CLASSIFICATION — FEATURE EXTRACTION

  FEATURE BLOCK 1 — TF-IDF  (word unigrams + bigrams)

  FEATURE BLOCK 2 — TF-IDF  (character n-grams 2-4)

  FEATURE BLOCK 3 — Word Embeddings  (mean-pooled)

  FEATURE BLOCK 4 — Combined  (TF-IDF + Embeddings)

  SAVING FEATURE EXTRACTION OUTPUTS

  FEATURE EXTRACTION SUMMARY
  Output directory    : /content/feature_extraction_output

  ── TF-IDF Word Features ──
  Vectorizer          : word, ngram=(1,2), max_features=10000
  X_train_tfidf       : (6545, 10000)
  X_test_tfidf        : (1981, 10000)

  ── TF-IDF Char Features ──
  Vectorizer          : char_wb, ngram=(2,4), max_features=5000
  X_train_tfidf_char  : (6545, 5000)
  X_test_tfidf_char   : (1981, 5000)

  ── Word Embeddings (mean-pooled) ──
  Embedding dim       : 100
  Source              : Word2Vec (trained on corpus)
  Normalisation       : L2
  X_train_embeddings  : (6545, 100)
  X_test_embed

In [6]:
"""
=============================================================================
  EMAIL SPAM CLASSIFICATION — MODEL TRAINING
  Project #3 : Email Spam Classification Using NLP and Machine Learning

  Trains three models on preprocessed Enron email data:
    1. Naive Bayes  (ComplementNB + Isotonic Calibration)
    2. SVM          (LinearSVC   + Sigmoid  Calibration)
    3. LSTM         (Bidirectional, 64-dim embeddings)

  Prerequisites (run in order before this script):
    → Preprocess.ipynb       produces  preprocessed_output/
    → feature_extraction.py  produces  feature_extraction_output/

  Outputs → trained_models/
    naive_bayes_model.pkl     Naive Bayes classifier
    svm_model.pkl             SVM classifier
    lstm_model.h5             LSTM Keras model
    lstm_tokenizer.pkl        Keras Tokenizer (word → int mapping)
    lstm_config.pkl           LSTM hyperparams (max_len, vocab_size, …)
    lstm_training_history.pkl Epoch-by-epoch loss / accuracy
    lstm_best_weights.h5      Best checkpoint saved by ModelCheckpoint
    tfidf_vectorizer.pkl      Shared TF-IDF vectorizer (for NB & SVM)

  Usage:
    python model_training.py

  Install dependencies:
    pip install scikit-learn numpy pandas scipy tensorflow
=============================================================================
"""

import os
import pickle
import shutil
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp

# ── Scikit-learn ──────────────────────────────────────────────────────────
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# ── TensorFlow / Keras ────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, LSTM, Dense, Dropout,
    Bidirectional, SpatialDropout1D,
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau,
)

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")


# =============================================================================
#  CONFIGURATION  — edit paths here if your files live elsewhere
# =============================================================================

PREPROCESS_DIR   = "preprocessed_output"       # output of Preprocess.ipynb
FEATURE_DIR      = "feature_extraction_output" # output of feature_extraction.py
MODEL_OUTPUT_DIR = "trained_models"            # destination for saved models

# ── LSTM hyper-parameters ─────────────────────────────────────────────────
VOCAB_SIZE  = 20_000   # max unique words kept by Keras Tokenizer
MAX_LEN     = 200      # all sequences padded / truncated to this length
EMBED_DIM   = 64       # word-embedding dimension
BATCH_SIZE  = 64
EPOCHS      = 10       # EarlyStopping will stop earlier if val_loss plateaus


# =============================================================================
#  HELPERS
# =============================================================================

def _find(candidates: list[str]) -> str:
    """Return the first existing path from *candidates*.

    Raises FileNotFoundError if none exist.
    """
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        f"Could not find any of: {candidates}\n"
        "Make sure Preprocess.ipynb and feature_extraction.py have been run first."
    )


def _banner(title: str) -> None:
    """Print a section banner to stdout."""
    print("\n" + "=" * 65)
    print(f"  {title}")
    print("=" * 65)


# =============================================================================
#  STEP 1 — LOAD DATA & FEATURES
# =============================================================================

def load_data():
    """Load all preprocessed artefacts needed for training.

    Returns
    -------
    df_train       : pd.DataFrame  — cleaned training rows (includes 'clean_text')
    df_test        : pd.DataFrame  — cleaned test rows
    X_train_tfidf  : sparse matrix — TF-IDF feature matrix for train
    X_test_tfidf   : sparse matrix — TF-IDF feature matrix for test
    y_train        : np.ndarray    — integer labels (0=ham, 1=spam)
    y_test         : np.ndarray
    tfidf_vectorizer : fitted TfidfVectorizer
    """
    _banner("STEP 1 — LOADING DATA & FEATURES")

    print("  Loading cleaned text DataFrames …")
    df_train = pd.read_csv(_find([
        f"{PREPROCESS_DIR}/train_cleaned.csv", "train_cleaned.csv",
    ]))
    df_test = pd.read_csv(_find([
        f"{PREPROCESS_DIR}/test_cleaned.csv", "test_cleaned.csv",
    ]))

    print("  Loading label arrays …")
    y_train = np.load(_find([
        f"{FEATURE_DIR}/y_train.npy", f"{PREPROCESS_DIR}/y_train.npy",
    ]))
    y_test = np.load(_find([
        f"{FEATURE_DIR}/y_test.npy", f"{PREPROCESS_DIR}/y_test.npy",
    ]))

    print("  Loading TF-IDF feature matrices …")
    X_train_tfidf = sp.load_npz(_find([f"{FEATURE_DIR}/X_train_tfidf.npz"]))
    X_test_tfidf  = sp.load_npz(_find([f"{FEATURE_DIR}/X_test_tfidf.npz"]))

    print("  Loading TF-IDF vectorizer …")
    with open(_find([f"{FEATURE_DIR}/tfidf_vectorizer.pkl"]), "rb") as fh:
        tfidf_vectorizer = pickle.load(fh)

    print(f"\n  Train rows          : {len(df_train)}")
    print(f"  Test  rows          : {len(df_test)}")
    print(f"  X_train_tfidf shape : {X_train_tfidf.shape}")
    print(f"  X_test_tfidf  shape : {X_test_tfidf.shape}")
    print(f"  y_train — ham: {(y_train == 0).sum()}  spam: {(y_train == 1).sum()}")
    print(f"  y_test  — ham: {(y_test  == 0).sum()}  spam: {(y_test  == 1).sum()}")

    return df_train, df_test, X_train_tfidf, X_test_tfidf, y_train, y_test, tfidf_vectorizer


# =============================================================================
#  STEP 2 — NAIVE BAYES
# =============================================================================

def train_naive_bayes(X_train, y_train):
    """Fit a calibrated ComplementNB classifier.

    ComplementNB handles class imbalance better than standard MultinomialNB.
    CalibratedClassifierCV wraps it so predict_proba() returns reliable
    probability estimates (needed for the Streamlit app).

    Parameters
    ----------
    X_train : sparse matrix  TF-IDF features (non-negative values required)
    y_train : np.ndarray     binary labels

    Returns
    -------
    nb_model : fitted CalibratedClassifierCV
    """
    _banner("STEP 2 — TRAINING NAIVE BAYES  (ComplementNB)")

    print("  Input shape  :", X_train.shape, " — sparse TF-IDF matrix")
    print("  Feature values are non-negative ✓  (required for Naive Bayes)")

    nb_base  = ComplementNB(alpha=0.1)          # alpha = Laplace smoothing
    nb_model = CalibratedClassifierCV(
        nb_base, cv=3, method="isotonic"        # isotonic regression calibration
    )

    print("\n  Fitting model on training data …")
    nb_model.fit(X_train, y_train)

    print("  Training complete!")
    print(f"  Model  : {type(nb_model).__name__} wrapping {type(nb_base).__name__}")
    print(f"  alpha  : 0.1  (Laplace smoothing)")
    print(f"  CV     : 3 folds  (isotonic calibration)")

    return nb_model


def save_naive_bayes(nb_model, output_dir: str) -> str:
    """Pickle the trained Naive Bayes model to *output_dir*."""
    path = os.path.join(output_dir, "naive_bayes_model.pkl")
    with open(path, "wb") as fh:
        pickle.dump(nb_model, fh)
    print(f"\n  Naive Bayes model saved → {path}")
    print(f"  File size : {os.path.getsize(path) / 1024:.1f} KB")
    return path


# =============================================================================
#  STEP 3 — SVM
# =============================================================================

def train_svm(X_train, y_train):
    """Fit a calibrated LinearSVC classifier.

    LinearSVC is fast on high-dimensional sparse TF-IDF features.
    class_weight='balanced' compensates for any class imbalance.
    CalibratedClassifierCV adds predict_proba() via sigmoid calibration.

    Parameters
    ----------
    X_train : sparse matrix  L2-normalised TF-IDF features
    y_train : np.ndarray     binary labels

    Returns
    -------
    svm_model : fitted CalibratedClassifierCV
    """
    _banner("STEP 3 — TRAINING SVM  (LinearSVC)")

    print("  Input shape  :", X_train.shape, " — L2-normalised TF-IDF matrix")
    print("  TF-IDF is already L2-normalised ✓  (good for SVM)")

    svm_base  = LinearSVC(
        C=1.0,                  # regularisation strength (higher = less regularised)
        class_weight="balanced",# adjusts weights inversely proportional to class frequencies
        max_iter=2000,          # ensure convergence on large datasets
        random_state=42,
    )
    svm_model = CalibratedClassifierCV(
        svm_base, cv=3, method="sigmoid"   # Platt scaling
    )

    print("\n  Fitting model on training data …")
    print("  (This may take a minute on large datasets)")
    svm_model.fit(X_train, y_train)

    print("  Training complete!")
    print(f"  Model        : {type(svm_model).__name__} wrapping {type(svm_base).__name__}")
    print(f"  C            : 1.0")
    print(f"  class_weight : balanced")
    print(f"  CV           : 3 folds  (sigmoid calibration)")

    return svm_model


def save_svm(svm_model, output_dir: str) -> str:
    """Pickle the trained SVM model to *output_dir*."""
    path = os.path.join(output_dir, "svm_model.pkl")
    with open(path, "wb") as fh:
        pickle.dump(svm_model, fh)
    print(f"\n  SVM model saved → {path}")
    print(f"  File size : {os.path.getsize(path) / 1024:.1f} KB")
    return path


# =============================================================================
#  STEP 4 — LSTM
# =============================================================================

def tokenize_for_lstm(train_texts, test_texts):
    """Build a Keras Tokenizer on training data and pad both splits.

    Steps
    -----
    1. Fit Tokenizer on training corpus only (avoids data leakage).
    2. Convert texts to integer sequences (unknown words → <OOV> token).
    3. Pad / truncate all sequences to MAX_LEN.

    Returns
    -------
    tokenizer        : fitted Keras Tokenizer
    vocab_size_actual: int  — capped at VOCAB_SIZE
    X_train_padded   : np.ndarray  shape (n_train, MAX_LEN)
    X_test_padded    : np.ndarray  shape (n_test,  MAX_LEN)
    """
    _banner("STEP 4a — LSTM TOKENIZATION")

    tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
    tokenizer.fit_on_texts(train_texts)

    vocab_size_actual = min(VOCAB_SIZE, len(tokenizer.word_index) + 1)

    print(f"  Vocabulary built from training corpus")
    print(f"  Total unique words  : {len(tokenizer.word_index):,}")
    print(f"  Vocabulary size used: {vocab_size_actual:,}  (capped at {VOCAB_SIZE:,})")

    print(f"\n  Converting texts to integer sequences …")
    X_train_seq = tokenizer.texts_to_sequences(train_texts)
    X_test_seq  = tokenizer.texts_to_sequences(test_texts)

    print(f"  Padding / truncating to length {MAX_LEN} …")
    X_train_padded = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
    X_test_padded  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding="post", truncating="post")

    print(f"\n  X_train_padded : {X_train_padded.shape}  (samples × sequence_length)")
    print(f"  X_test_padded  : {X_test_padded.shape}")

    return tokenizer, vocab_size_actual, X_train_padded, X_test_padded


def build_lstm_model(vocab_size_actual: int) -> tf.keras.Model:
    """Construct and compile the Bidirectional LSTM model.

    Architecture
    ------------
    Input (MAX_LEN tokens)
      → Embedding        (vocab_size × EMBED_DIM)
      → SpatialDropout1D (0.2)
      → Bidirectional LSTM (64 units, dropout=0.2)
      → Dense            (128, ReLU)
      → Dropout          (0.3)
      → Dense            (1, sigmoid)  →  spam probability

    Returns
    -------
    lstm_model : compiled tf.keras.Model
    """
    _banner("STEP 4b — BUILD LSTM ARCHITECTURE")

    tf.random.set_seed(42)
    np.random.seed(42)

    model = Sequential([
        # Converts integer token IDs to dense EMBED_DIM-dimensional vectors
        Embedding(
            input_dim=vocab_size_actual,
            output_dim=EMBED_DIM,
            input_length=MAX_LEN,
            name="embedding",
        ),
        # Drops entire embedding vectors (more effective than per-element dropout for text)
        SpatialDropout1D(0.2, name="spatial_dropout"),
        # Reads the sequence both left→right and right→left
        Bidirectional(
            LSTM(64, dropout=0.2, recurrent_dropout=0.2),
            name="bilstm",
        ),
        Dense(128, activation="relu", name="dense_1"),
        Dropout(0.3, name="dropout"),
        # Single sigmoid output: > 0.5 → spam, < 0.5 → ham
        Dense(1, activation="sigmoid", name="output"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    model.summary()
    return model


def get_lstm_callbacks(output_dir: str) -> list:
    """Create training callbacks.

    EarlyStopping      — halts training when val_loss stops improving (patience=3)
    ModelCheckpoint    — saves the epoch with the lowest val_loss
    ReduceLROnPlateau  — halves the learning rate when val_loss stagnates (patience=2)
    """
    checkpoint_path = os.path.join(output_dir, "lstm_best_weights.h5")

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True,
            verbose=1,
        ),
        ModelCheckpoint(
            filepath=checkpoint_path,
            monitor="val_loss",
            save_best_only=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

    print(f"\n  Callbacks configured:")
    print(f"  EarlyStopping     — patience=3,  monitors val_loss")
    print(f"  ModelCheckpoint   — best weights → {checkpoint_path}")
    print(f"  ReduceLROnPlateau — factor=0.5,  patience=2")

    return callbacks


def train_lstm(model, X_train_padded, y_train, callbacks):
    """Run the LSTM training loop.

    Parameters
    ----------
    model           : compiled Keras model
    X_train_padded  : np.ndarray  padded token sequences
    y_train         : np.ndarray  binary labels
    callbacks       : list        Keras callbacks

    Returns
    -------
    history : Keras History object
    """
    _banner("STEP 4c — TRAINING LSTM")

    print(f"  Training samples : {len(X_train_padded)}")
    print(f"  Batch size       : {BATCH_SIZE}")
    print(f"  Max epochs       : {EPOCHS}  (EarlyStopping may end sooner)")
    print(f"  Validation split : 20% of training data\n")

    history = model.fit(
        X_train_padded,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        callbacks=callbacks,
        verbose=1,
    )

    epochs_done   = len(history.history["loss"])
    best_val_loss = min(history.history["val_loss"])
    best_val_acc  = max(history.history["val_accuracy"])

    print(f"\n  Training complete!")
    print(f"  Epochs trained    : {epochs_done}")
    print(f"  Best val loss     : {best_val_loss:.4f}")
    print(f"  Best val accuracy : {best_val_acc:.4f}")

    return history


def save_lstm(model, tokenizer, vocab_size_actual, history, output_dir: str):
    """Persist the LSTM model, tokenizer, config, and training history."""
    _banner("STEP 4d — SAVING LSTM ARTEFACTS")

    # Full Keras model (architecture + weights)
    model_path = os.path.join(output_dir, "lstm_model.h5")
    model.save(model_path)
    print(f"  LSTM model saved          → {model_path}")

    # Tokenizer  (word → integer mapping, required in Streamlit app)
    tokenizer_path = os.path.join(output_dir, "lstm_tokenizer.pkl")
    with open(tokenizer_path, "wb") as fh:
        pickle.dump(tokenizer, fh)
    print(f"  LSTM tokenizer saved      → {tokenizer_path}")

    # Config dict  (needed to reconstruct pad_sequences in Streamlit)
    config = {
        "vocab_size": vocab_size_actual,
        "max_len"   : MAX_LEN,
        "embed_dim" : EMBED_DIM,
    }
    config_path = os.path.join(output_dir, "lstm_config.pkl")
    with open(config_path, "wb") as fh:
        pickle.dump(config, fh)
    print(f"  LSTM config saved         → {config_path}")

    # Training history  (for evaluation / plotting notebook)
    history_path = os.path.join(output_dir, "lstm_training_history.pkl")
    with open(history_path, "wb") as fh:
        pickle.dump(history.history, fh)
    print(f"  Training history saved    → {history_path}")


# =============================================================================
#  STEP 5 — SAVE SHARED ARTEFACTS
# =============================================================================

def save_shared_artefacts(tfidf_vectorizer, output_dir: str):
    """Copy the shared TF-IDF vectorizer into trained_models/.

    Both Naive Bayes and SVM require this to transform raw text at
    inference time in the Streamlit app.
    """
    _banner("STEP 5 — SAVING SHARED ARTEFACTS")

    dest = os.path.join(output_dir, "tfidf_vectorizer.pkl")
    with open(dest, "wb") as fh:
        pickle.dump(tfidf_vectorizer, fh)
    print(f"  TF-IDF vectorizer saved → {dest}")
    print("  (Shared by Naive Bayes and SVM)")


# =============================================================================
#  STEP 6 — SUMMARY
# =============================================================================

def print_summary(output_dir: str):
    """Print a table of all saved files with sizes."""
    _banner("STEP 6 — TRAINING SUMMARY")

    saved_files = [
        ("naive_bayes_model.pkl",     "Naive Bayes  (ComplementNB + Calibration)"),
        ("svm_model.pkl",             "SVM          (LinearSVC   + Calibration)"),
        ("lstm_model.h5",             "LSTM         (Bidirectional, 64-dim)"),
        ("lstm_tokenizer.pkl",        "LSTM Tokenizer  (word → int mapping)"),
        ("lstm_config.pkl",           "LSTM Config     (vocab_size, max_len, …)"),
        ("lstm_training_history.pkl", "LSTM Training History"),
        ("lstm_best_weights.h5",      "LSTM Best Checkpoint"),
        ("tfidf_vectorizer.pkl",      "Shared TF-IDF Vectorizer  (NB + SVM)"),
    ]

    print(f"\n  Output directory : {os.path.abspath(output_dir)}/\n")
    for fname, desc in saved_files:
        fpath = os.path.join(output_dir, fname)
        if os.path.exists(fpath):
            size_kb = os.path.getsize(fpath) / 1024
            print(f"  ✓  {fname:<40}  {size_kb:>8.1f} KB  — {desc}")
        else:
            print(f"  ✗  {fname:<40}  NOT FOUND")

    print("""
  ── How to load models in the Streamlit app ───────────────────────

  import pickle, tensorflow as tf
  from tensorflow.keras.preprocessing.sequence import pad_sequences

  # Naive Bayes
  nb    = pickle.load(open("trained_models/naive_bayes_model.pkl", "rb"))

  # SVM
  svm   = pickle.load(open("trained_models/svm_model.pkl", "rb"))

  # LSTM
  lstm  = tf.keras.models.load_model("trained_models/lstm_model.h5")
  tok   = pickle.load(open("trained_models/lstm_tokenizer.pkl", "rb"))
  cfg   = pickle.load(open("trained_models/lstm_config.pkl",    "rb"))

  # TF-IDF vectorizer  (for Naive Bayes & SVM)
  tfidf = pickle.load(open("trained_models/tfidf_vectorizer.pkl", "rb"))
""")


# =============================================================================
#  MAIN
# =============================================================================

def main():
    print("=" * 65)
    print("  EMAIL SPAM CLASSIFICATION — MODEL TRAINING")
    print("  Project #3")
    print("=" * 65)
    print(f"  TensorFlow version : {tf.__version__}")
    print(f"  NumPy version      : {np.__version__}")

    # ── Create output directory ───────────────────────────────────────────
    os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
    print(f"\n  Models will be saved to: {os.path.abspath(MODEL_OUTPUT_DIR)}")

    # ── 1. Load data ──────────────────────────────────────────────────────
    (
        df_train, df_test,
        X_train_tfidf, X_test_tfidf,
        y_train, y_test,
        tfidf_vectorizer,
    ) = load_data()

    train_texts = df_train["clean_text"].fillna("").values
    test_texts  = df_test["clean_text"].fillna("").values

    # ── 2. Naive Bayes ────────────────────────────────────────────────────
    nb_model = train_naive_bayes(X_train_tfidf, y_train)
    save_naive_bayes(nb_model, MODEL_OUTPUT_DIR)

    # ── 3. SVM ────────────────────────────────────────────────────────────
    svm_model = train_svm(X_train_tfidf, y_train)
    save_svm(svm_model, MODEL_OUTPUT_DIR)

    # ── 4. LSTM ───────────────────────────────────────────────────────────
    tokenizer, vocab_size_actual, X_train_padded, X_test_padded = tokenize_for_lstm(
        train_texts, test_texts
    )
    lstm_model = build_lstm_model(vocab_size_actual)
    callbacks  = get_lstm_callbacks(MODEL_OUTPUT_DIR)
    history    = train_lstm(lstm_model, X_train_padded, y_train, callbacks)
    save_lstm(lstm_model, tokenizer, vocab_size_actual, history, MODEL_OUTPUT_DIR)

    # ── 5. Shared artefacts ───────────────────────────────────────────────
    save_shared_artefacts(tfidf_vectorizer, MODEL_OUTPUT_DIR)

    # ── 6. Summary ────────────────────────────────────────────────────────
    print_summary(MODEL_OUTPUT_DIR)

    print("\n  All models trained and saved successfully ✓\n")


if __name__ == "__main__":
    main()


  EMAIL SPAM CLASSIFICATION — MODEL TRAINING
  Project #3
  TensorFlow version : 2.20.0
  NumPy version      : 2.0.2

  Models will be saved to: /content/trained_models

  STEP 1 — LOADING DATA & FEATURES
  Loading cleaned text DataFrames …
  Loading label arrays …
  Loading TF-IDF feature matrices …
  Loading TF-IDF vectorizer …

  Train rows          : 6545
  Test  rows          : 1981
  X_train_tfidf shape : (6545, 10000)
  X_test_tfidf  shape : (1981, 10000)
  y_train — ham: 3294  spam: 3251
  y_test  — ham: 989  spam: 992

  STEP 2 — TRAINING NAIVE BAYES  (ComplementNB)
  Input shape  : (6545, 10000)  — sparse TF-IDF matrix
  Feature values are non-negative ✓  (required for Naive Bayes)

  Fitting model on training data …
  Training complete!
  Model  : CalibratedClassifierCV wrapping ComplementNB
  alpha  : 0.1  (Laplace smoothing)
  CV     : 3 folds  (isotonic calibration)

  Naive Bayes model saved → trained_models/naive_bayes_model.pkl
  File size : 1175.4 KB

  STEP 3 — TRAIN

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout                 │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm (Bidirectional)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

...


  Callbacks configured:
  EarlyStopping     — patience=3,  monitors val_loss
  ModelCheckpoint   — best weights → trained_models/lstm_best_weights.h5
  ReduceLROnPlateau — factor=0.5,  patience=2

  STEP 4c — TRAINING LSTM
  Training samples : 6545
  Batch size       : 64
  Max epochs       : 10  (EarlyStopping may end sooner)
  Validation split : 20% of training data

Epoch 1/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6818 - loss: 0.5464
Epoch 1: val_loss improved from None to 0.08517, saving model to trained_models/lstm_best_weights.h5



Epoch 1: finished saving model to trained_models/lstm_best_weights.h5
82/82 ━━━━━━━━━━━━━━━━━━━━ 131s 1s/step - accuracy: 0.8340 - loss: 0.3465 - val_accuracy: 0.9694 - val_loss: 0.0852 - learning_rate: 0.0010
Epoch 2/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9836 - loss: 0.0545
Epoch 2: val_loss improved from 0.08517 to 0.08021, saving model to trained_models/lstm_best_weights.h5



Epoch 2: finished saving model to trained_models/lstm_best_weights.h5
82/82 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - accuracy: 0.9855 - loss: 0.0482 - val_accuracy: 0.9702 - val_loss: 0.0802 - learning_rate: 0.0010
Epoch 3/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9924 - loss: 0.0285
Epoch 3: val_loss improved from 0.08021 to 0.07352, saving model to trained_models/lstm_best_weights.h5



Epoch 3: finished saving model to trained_models/lstm_best_weights.h5
82/82 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - accuracy: 0.9933 - loss: 0.0252 - val_accuracy: 0.9771 - val_loss: 0.0735 - learning_rate: 0.0010
Epoch 4/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9966 - loss: 0.0128
Epoch 4: val_loss did not improve from 0.07352
82/82 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - accuracy: 0.9962 - loss: 0.0161 - val_accuracy: 0.9740 - val_loss: 0.0913 - learning_rate: 0.0010
Epoch 5/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9974 - loss: 0.0149
Epoch 5: val_loss did not improve from 0.07352

Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
82/82 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - accuracy: 0.9977 - loss: 0.0097 - val_accuracy: 0.9748 - val_loss: 0.0847 - learning_rate: 0.0010
Epoch 6/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9986 - loss: 0.0049
Epoch 6: val_loss did not improve from 0.07352
82/82 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - 


  Training complete!
  Epochs trained    : 6
  Best val loss     : 0.0735
  Best val accuracy : 0.9771

  STEP 4d — SAVING LSTM ARTEFACTS
  LSTM model saved          → trained_models/lstm_model.h5
  LSTM tokenizer saved      → trained_models/lstm_tokenizer.pkl
  LSTM config saved         → trained_models/lstm_config.pkl
  Training history saved    → trained_models/lstm_training_history.pkl

  STEP 5 — SAVING SHARED ARTEFACTS
  TF-IDF vectorizer saved → trained_models/tfidf_vectorizer.pkl
  (Shared by Naive Bayes and SVM)

  STEP 6 — TRAINING SUMMARY

  Output directory : /content/trained_models/

  ✓  naive_bayes_model.pkl                       1175.4 KB  — Naive Bayes  (ComplementNB + Calibration)
  ✓  svm_model.pkl                                236.0 KB  — SVM          (LinearSVC   + Calibration)
  ✓  lstm_model.h5                              16019.1 KB  — LSTM         (Bidirectional, 64-dim)
  ✓  lstm_tokenizer.pkl                          2145.0 KB  — LSTM Tokenizer  (word → int